# Modelado – Detección temprana de riesgo académico – UNCAus
**Doctorado en Informática – UTN FRRe/UNNE/UNAM**  
Curso: Introducción al Aprendizaje de Máquina  
Integrantes: Stella Gerzel · Juan Manuel Goya · Gustavo Ebel

---
Este notebook entrena y compara tres algoritmos sobre los 3 problemas de clasificación:
- Regresión Logística
- Random Forest  
- XGBoost

Usando validación cruzada 5-fold estratificada dentro del train.
El test se usa UNA SOLA VEZ al final para la evaluación definitiva.

In [ ]:
# CELDA 1 — Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110
print('Librerías OK')

In [ ]:
# CELDA 2 — Subir y preparar los datos (igual que notebooks anteriores)
from google.colab import files
uploaded = files.upload()

def encontrar(nombres, *palabras):
    for n in nombres:
        if all(p.lower() in n.lower() for p in palabras):
            return n
    return None

nombres = list(uploaded.keys())
reglas  = [
    (('sistem','2024'), 'Result_1_2', 'Sistemas y Organizaciones',        2024),
    (('sistem','2025'), 'Result_8_2', 'Sistemas y Organizaciones',        2025),
    (('algor', '2024'), 'Result_4_2', 'Algoritmos y Estructuras de Datos', 2024),
    (('algor', '2025'), 'Result_5_2', 'Algoritmos y Estructuras de Datos', 2025),
]
dfs = []
for palabras, hoja, materia, cohorte in reglas:
    archivo = encontrar(nombres, *palabras)
    if not archivo: continue
    tmp = pd.read_excel(io.BytesIO(uploaded[archivo]), sheet_name=hoja)
    tmp.insert(0,'materia', materia); tmp.insert(1,'cohorte', cohorte)
    dfs.append(tmp)
df = pd.concat(dfs, ignore_index=True)

# Normalizar faltantes
df_w = df.copy()
for col in df_w.select_dtypes('object').columns:
    df_w[col] = df_w[col].replace(['-','_',' '], np.nan)

# Descartar columnas (mismos criterios del EDA)
pct_miss  = df_w.isna().mean()
vacias    = pct_miss[pct_miss == 1.0].index.tolist()
sin_var   = df_w.columns[df_w.nunique() <= 1].tolist()
descartar = list(set(vacias + sin_var +
    ['procedencia_calle','procedencia_numero','procedencia_piso',
     'procedencia_departamento','procedencia_unidad',
     'procedencia_barrio','procedencia_codigo_postal',
     'trabajo_tarea_descripcion','costeo_estudios_descripcion',
     'idioma_otro','nivel_idioma_otro','alumno','nota']))

columnas_modelo = [c for c in df_w.columns if c not in descartar]
X_base = df_w[columnas_modelo].copy()

# 3 targets
y1 = df_w['nota'].map({'A':0,'R':1,'U':1})
mask_p2 = df_w['nota'].isin(['A','R'])
X_p2 = X_base[mask_p2].reset_index(drop=True)
y2   = df_w.loc[mask_p2,'nota'].map({'A':0,'R':1}).reset_index(drop=True)
mask_p3 = df_w['nota'].isin(['A','U'])
X_p3 = X_base[mask_p3].reset_index(drop=True)
y3   = df_w.loc[mask_p3,'nota'].map({'A':0,'U':1}).reset_index(drop=True)

# Splits
X1_tr,X1_te,y1_tr,y1_te = train_test_split(X_base,y1,test_size=0.25,stratify=y1,random_state=42)
X2_tr,X2_te,y2_tr,y2_te = train_test_split(X_p2,  y2,test_size=0.25,stratify=y2,random_state=42)
X3_tr,X3_te,y3_tr,y3_te = train_test_split(X_p3,  y3,test_size=0.25,stratify=y3,random_state=42)

# Preprocesador
cols_cat = X_base.select_dtypes('object').columns.tolist()
cols_num = X_base.select_dtypes('number').columns.tolist()
preprocesador = ColumnTransformer([
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cols_cat),
    ('num', SimpleImputer(strategy='most_frequent'), cols_num)
])

print(f'Datos listos. Columnas al modelo: {len(columnas_modelo)}')
print(f'Train P1: {len(X1_tr)} | Train P2: {len(X2_tr)} | Train P3: {len(X3_tr)}')

---
## Validación cruzada 5-fold — comparar algoritmos dentro del train

In [ ]:
# CELDA 3 — Definir los algoritmos y correr validación cruzada
#
# StratifiedKFold: validación cruzada que mantiene la proporción de clases
# en cada fold (estratificada), igual que el split train/test
#
# scoring='roc_auc': métrica principal para comparar
# ROC-AUC mide qué tan bien distingue el modelo entre las dos clases
# 0.5 = al azar | 1.0 = perfecto

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

algoritmos = {
    'LogReg': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'RF':     RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42),
    'XGB':    XGBClassifier(scale_pos_weight=1, eval_metric='logloss',
                            use_label_encoder=False, random_state=42, verbosity=0)
}

problemas = {
    'P1 Riesgo global':  (X1_tr, y1_tr),
    'P2 Desaprobacion':  (X2_tr, y2_tr),
    'P3 Abandono':       (X3_tr, y3_tr),
}

print('Corriendo validación cruzada 5-fold (puede tardar ~1 minuto)...\n')
resultados = {}

for nombre_p, (X_tr, y_tr) in problemas.items():
    resultados[nombre_p] = {}
    for nombre_a, modelo in algoritmos.items():
        pipe = Pipeline([('prep', preprocesador), ('modelo', modelo)])
        scores = cross_val_score(pipe, X_tr, y_tr, cv=cv,
                                 scoring='roc_auc', n_jobs=-1)
        resultados[nombre_p][nombre_a] = scores
        print(f'  {nombre_p} | {nombre_a}: '
              f'ROC-AUC = {scores.mean():.3f} ± {scores.std():.3f}')
    print()

In [ ]:
# CELDA 4 — Visualizar comparación de algoritmos

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (nombre_p, scores_dict) in zip(axes, resultados.items()):
    nombres_alg = list(scores_dict.keys())
    medias = [scores_dict[a].mean() for a in nombres_alg]
    stds   = [scores_dict[a].std()  for a in nombres_alg]
    colores_bar = ['#3498DB','#2ECC71','#E74C3C']
    bars = ax.bar(nombres_alg, medias, yerr=stds, capsize=6,
                  color=colores_bar, alpha=0.85, edgecolor='white')
    ax.set_ylim(0.4, 1.0)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Azar (0.5)')
    ax.set_title(nombre_p, fontweight='bold')
    ax.set_ylabel('ROC-AUC (5-fold CV)')
    ax.legend(fontsize=8)
    for bar, media, std in zip(bars, medias, stds):
        ax.text(bar.get_x()+bar.get_width()/2, media+std+0.01,
                f'{media:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Comparación de algoritmos por ROC-AUC (validación cruzada 5-fold)',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Las barras de error muestran la variabilidad entre los 5 folds.')
print('Un modelo con alta media y bajo desvío es más confiable.')

---
## Evaluación final sobre el test — UNA SOLA VEZ

In [ ]:
# CELDA 5 — Entrenar el mejor modelo sobre TODO el train y evaluar en test
#
# Usamos Random Forest para los 3 problemas (mejor resultado en el paper
# de referencia y generalmente robusto)
# Cambiá el algoritmo si la validación cruzada muestra otro ganador

mejor_modelo = RandomForestClassifier(class_weight='balanced',
                                      n_estimators=100, random_state=42)

splits_test = [
    ('P1 Riesgo global', X1_tr, X1_te, y1_tr, y1_te),
    ('P2 Desaprobacion', X2_tr, X2_te, y2_tr, y2_te),
    ('P3 Abandono',      X3_tr, X3_te, y3_tr, y3_te),
]

pipes_entrenados = {}
print('Entrenando sobre todo el train y evaluando en test...\n')

for nombre, X_tr, X_te, y_tr, y_te in splits_test:
    pipe = Pipeline([('prep', preprocesador), ('modelo', mejor_modelo)])
    pipe.fit(X_tr, y_tr)
    pipes_entrenados[nombre] = pipe

    y_pred      = pipe.predict(X_te)
    y_prob      = pipe.predict_proba(X_te)[:,1]
    auc         = roc_auc_score(y_te, y_prob)

    print(f'── {nombre} ──')
    print(f'  ROC-AUC en test: {auc:.3f}')
    print(classification_report(y_te, y_pred,
                                target_names=['Aprobó (0)','Riesgo (1)']))
    print()

In [ ]:
# CELDA 6 — Matrices de confusión y curvas ROC

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for col, (nombre, X_tr, X_te, y_tr, y_te) in enumerate(splits_test):
    pipe = pipes_entrenados[nombre]

    # Fila 0: Matriz de confusión
    ConfusionMatrixDisplay.from_estimator(
        pipe, X_te, y_te,
        display_labels=['Aprobó','Riesgo'],
        colorbar=False, ax=axes[0, col])
    axes[0, col].set_title(f'{nombre}\nMatriz de confusión', fontweight='bold')

    # Fila 1: Curva ROC
    RocCurveDisplay.from_estimator(pipe, X_te, y_te, ax=axes[1, col])
    axes[1, col].set_title(f'{nombre}\nCurva ROC', fontweight='bold')
    axes[1, col].plot([0,1],[0,1],'--',color='gray',alpha=0.5,label='Azar')
    axes[1, col].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Próximo paso: ajuste de umbral e importancia de variables

In [ ]:
# CELDA 7 — Anticipo: qué pasa si movemos el umbral de decisión
#
# Por defecto el modelo predice clase 1 si probabilidad > 0.50
# Bajando el umbral aumentamos el recall (detectamos más casos reales)
# a costa de más falsas alarmas

from sklearn.metrics import recall_score, precision_score

pipe_p1  = pipes_entrenados['P1 Riesgo global']
y_prob_p1 = pipe_p1.predict_proba(X1_te)[:,1]

umbrales = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]
filas = []
for u in umbrales:
    y_pred_u = (y_prob_p1 >= u).astype(int)
    filas.append({
        'Umbral':     u,
        'Recall':     recall_score(y1_te, y_pred_u),
        'Precision':  precision_score(y1_te, y_pred_u),
        'Falsas alarmas': int((y_pred_u == 1).sum() - (y1_te[y_pred_u==1]==1).sum())
    })

tabla_umbrales = pd.DataFrame(filas).set_index('Umbral')
print('P1 – Efecto de mover el umbral de decisión:')
print(tabla_umbrales.round(3).to_string())
print()
print('Recall sube → detectamos más casos reales de riesgo')
print('Precision baja → más falsas alarmas')
print('La elección del umbral es una decisión institucional, no matemática.')